# Site-Specific DUIDD Finetuning Tutorial

This notebook provides a guide to the pretraining and site-specific finetuning workflow in the `nrx_duidd` repository, as described in [[1]](#ref-1) and [[2]](#ref-2).

The command cells below describe operations that can be computationally expensive. They are intentionally not executed automatically, and only the commands themselves are printed. Run these commands in a shell rather than from the notebook. Avoid other GPU-intensive workloads while the scripts are running, as GPU load can affect the results.


## Contents

- [1. Prerequisites and repository paths](#prerequisites)
- [2. The configuration parameters](#configuration)
- [3. Pretrain DUIDD on synthetic channels](#pretraining)
- [4. Prepare the finetuning TFRecord](#tfrecord-preparation)
- [5. Finetune with the Data Lake Channel](#finetuning)
- [6. Evaluate on real-world data](#evaluation)
- [7. Examine the results](#results)
- [References](#references)


<a id="prerequisites"></a>

## 1. Prerequisites and Repository Paths

The required packages are listed in [`requirements.txt`](../requirements.txt). Check [`README.md`](../README.md) for the complete information.

Real-data steps require a running ClickHouse-backed Aerial Data Lake, databases containing the required `fapi` and `fh` tables, and NVIDIA pyAerial. The databases are not included in the repository; detailed information and download instructions are provided in [`datasets.md`](../../datasets.md).


In [2]:
from pathlib import Path
import os

candidates = (Path.cwd(), *Path.cwd().parents)
WORKSPACE_ROOT = next((path.resolve() for path in candidates if path.name == 'rx_training' and (path / 'nrx_duidd' / 'config').is_dir() and (path / 'nrx_duidd' / 'scripts').is_dir()), None)
if WORKSPACE_ROOT is None:
    raise RuntimeError('Could not locate the rx_training workspace root.')

os.chdir(WORKSPACE_ROOT)
REPO = Path('nrx_duidd')
DISPLAY_ROOT = Path('/rx_training')

print(f'Workspace root: {DISPLAY_ROOT}')
print(f'Repository: {DISPLAY_ROOT / REPO}')


Workspace root: /rx_training
Repository: /rx_training/nrx_duidd


<a id="configuration"></a>

## 2. The Configuration Parameters

Every experiment is driven by a file in `config/`. Its `global.label` becomes the prefix for weight checkpoints and covariance files. The DUIDD schedule `[N_1, ..., N_I]` gives the number of flooding min-sum LDPC message-passing iterations after each of the `I` MMSE-PIC detection stages. All configurations used in this work keep the total number of decoding iterations fixed at 12.

The pretraining and finetuning configurations use the `[6, 6]` schedule. Pretraining uses synthetic 3GPP UMi channels. Finetuning uses `channel_type = 'Datalake'`, with transmission parameters fixed to MCS index 10, and one fixed scrambling configuration.

An uplink layer (ULL) is one active spatial transmission layer. The DUIDD finetuning experiment uses the 2 ULL dataset. The tables in [`datasets.md`](../../datasets.md) and the [DUIDD README](../README.md) list all configurations used for DUIDD, classical IDD, and LMMSE channel-estimation evaluation.


The following code snippet selects the DUIDD pretraining configuration and the finetuning configuration for the Jun. 2026 Small Laboratory dual-layer dataset.


In [3]:
from nrx_duidd.notebooks.notebook_helpers import NotebookHelpers

notebook = NotebookHelpers(WORKSPACE_ROOT, DISPLAY_ROOT)

pretraining_config = 'duidd_aerial_6_6.cfg'
finetuning_config = 'duidd_pixel9pro_j61_2ULL_slot14_mcs10.cfg'

pretraining_config_path = REPO / 'config' / pretraining_config
finetuning_config_path = REPO / 'config' / finetuning_config

pretraining_label = notebook.config_value(pretraining_config_path, 'label')
finetuning_label = notebook.config_value(finetuning_config_path, 'label')
finetuning_dataset = notebook.config_value(finetuning_config_path, 'datalake_tf_fn')

notebook.show_config(pretraining_config_path)
notebook.show_config(finetuning_config_path)


--- /rx_training/nrx_duidd/config/duidd_aerial_6_6.cfg

label = 'duidd_aerial_6_6' # all relevant files such as weights will use this label
n_size_bwp = 4
mcs_index = [14]
symbol_allocation = [0, 13]
n_rntis = [1, 1]
n_ids = [1, 1]
num_layers = 1
chest = "lslin"
duidd_schedule = [6, 6]
num_iter_train_save = 1000
channel_type = 'UMi'

--- /rx_training/nrx_duidd/config/duidd_pixel9pro_j61_2ULL_slot14_mcs10.cfg

label = 'duidd_pixel9pro_j61_2ULL_slot14_mcs10'
n_size_bwp = 273
mcs_index = [10]
symbol_allocation = [0, 13]
n_rntis = [9304, 9304]
n_ids = [51, 51]
num_layers = 2
chest = "lslin"
duidd_schedule = [6, 6]
num_iter_train_save = 100
channel_type = 'Datalake'
datalake_tf_fn = 'Pixel9Pro_2ULLs_12dB_j61_2026_06_04_no_pyaerial_label.tfrecord'



<a id="pretraining"></a>

## 3. Pretrain DUIDD on Synthetic Channels

Pretraining initializes the receiver before site-specific finetuning. Run [`train_neural_rx.py`](../scripts/train_neural_rx.py) with `duidd_aerial_6_6.cfg`. The configuration trains on dual-layer synthetic 3GPP UMi channels for 6,000 batches: 3,000 batches with binary cross-entropy (BCE) loss followed by 3,000 batches with normalized LogSumExp BLER loss.

The trainer loads `weights/<label>_weights` when that file already exists; otherwise it starts from random weights. Training saves the current weights to [`/rx_training/nrx_duidd/weights/`](../weights/) after every 1,000 batches and writes TensorBoard logs to a `<label>-<timestamp>/` directory under [`/rx_training/nrx_duidd/logs/`](../logs/). The provided pretrained weights can be used when reproducing the workflow.


In [4]:
pretrained_weights = REPO / 'weights' / f'{pretraining_label}_weights'
pretraining_command = f'''    cd {notebook.display_path(REPO / 'scripts')}
python train_neural_rx.py -config_name {pretraining_config}
'''.strip()

notebook.show_terminal(
    pretraining_command,
    [('Weights to be saved to', notebook.display_path(pretrained_weights))],
)
# os.system(pretraining_command)


```bash
$ cd /rx_training/nrx_duidd/scripts
$ python train_neural_rx.py -config_name duidd_aerial_6_6.cfg
```

```text
Weights to be saved to: /rx_training/nrx_duidd/weights/duidd_aerial_6_6_weights
```

<a id="tfrecord-preparation"></a>

## 4. Prepare the Finetuning TFRecord

DUIDD finetuning uses the provided Data Lake-derived TFRecord. Place it in [`/rx_training/finetuning_datasets/`](../../finetuning_datasets/) using the filename selected by `datalake_tf_fn`. The file contains the samples and labels required during DUIDD finetuning.

The DUIDD dataset is restricted to slot-14, MCS index 10, dual-layer transmission, with one scrambling configuration. This restriction is required because DUIDD includes LDPC decoding during training and its information-bit labels and descrambling must match the configured transport block. The provided file can be used directly.


In [5]:
dataset_path = REPO.parent / 'finetuning_datasets' / finetuning_dataset

print(f'Configured TFRecord: {notebook.display_path(dataset_path)}')
print(f'Exists: {notebook.resolve(dataset_path).is_file()}')


Configured TFRecord: /rx_training/finetuning_datasets/Pixel9Pro_2ULLs_12dB_j61_2026_06_04_no_pyaerial_label.tfrecord
Exists: False


<a id="finetuning"></a>

## 5. Finetune with the Data Lake Channel

Finetuning reuses [`train_neural_rx.py`](../scripts/train_neural_rx.py). The selected configuration sets `channel_type = 'Datalake'` and points `datalake_tf_fn` to the TFRecord from the previous step.

**Important:** Finetuning expects a weight file matching `global.label`. Copy the pretrained `[6, 6]` weights to the finetuning label before starting.

The finetuning configuration runs for 2,000 batches. The first 1,000 batches use BCE loss, and the next 1,000 use normalized LogSumExp BLER loss. Weights are saved every 100 batches. TensorBoard logs are written to a `<label>-<timestamp>/` directory under [`/rx_training/nrx_duidd/logs/`](../logs/).


In [6]:
finetuned_weights = REPO / 'weights' / f'{finetuning_label}_weights'
finetuning_command = f'''    cp {notebook.display_path(pretrained_weights)} {notebook.display_path(finetuned_weights)}
cd {notebook.display_path(REPO / 'scripts')}
python train_neural_rx.py -config_name {finetuning_config}
'''.strip()

notebook.show_terminal(
    finetuning_command,
    [
        ('Finetuning dataset', notebook.display_path(dataset_path)),
        ('Weights to be saved to', notebook.display_path(finetuned_weights)),
    ],
)
# os.system(finetuning_command)


```bash
$ cp /rx_training/nrx_duidd/weights/duidd_aerial_6_6_weights /rx_training/nrx_duidd/weights/duidd_pixel9pro_j61_2ULL_slot14_mcs10_weights
$ cd /rx_training/nrx_duidd/scripts
$ python train_neural_rx.py -config_name duidd_pixel9pro_j61_2ULL_slot14_mcs10.cfg
```

```text
Finetuning dataset: /rx_training/finetuning_datasets/Pixel9Pro_2ULLs_12dB_j61_2026_06_04_no_pyaerial_label.tfrecord
Weights to be saved to: /rx_training/nrx_duidd/weights/duidd_pixel9pro_j61_2ULL_slot14_mcs10_weights
```

<a id="evaluation"></a>

## 6. Evaluate the Model on Real-World Data

[`eval_duidd_from_datalake.py`](../scripts/eval_duidd_from_datalake.py) retrieves the selected slot samples from ClickHouse and evaluates DUIDD and the MMSE Reference Rx on the same samples. The MMSE Reference Rx corresponds to the pyAerial PUSCH Rx presented in [[3]](#ref-3). The DUIDD receiver loads `weights/<label>_weights` from the selected configuration.

For deterministic evaluation, pass a timestamp pickle from [`/rx_training/nrx_duidd/eval_timestamps/`](../eval_timestamps/). Dataset BLER is the fraction of transport blocks that fail after offline receiver processing and LDPC decoding on the fixed test dataset.

Results are printed to the console and appended to `/rx_training/nrx_duidd/scripts/results_duidd_datalake.txt`.


In [7]:
evaluation_results = REPO / 'scripts' / 'results_duidd_datalake.txt'
evaluation_command = f'''    cd {notebook.display_path(REPO / 'scripts')}
python eval_duidd_from_datalake.py --config-name {finetuning_config} --db 8 --timestamps slot1000.pkl
'''.strip()

notebook.show_terminal(
    evaluation_command,
    [('Expected results log', notebook.display_path(evaluation_results))],
)
# os.system(evaluation_command)


```bash
$ cd /rx_training/nrx_duidd/scripts
$ python eval_duidd_from_datalake.py --config-name duidd_pixel9pro_j61_2ULL_slot14_mcs10.cfg --db 8 --timestamps slot1000.pkl
```

```text
Expected results log: /rx_training/nrx_duidd/scripts/results_duidd_datalake.txt
```

<a id="results"></a>

## 7. Examine the Results

On the Jun. 2026 dual-layer Small Laboratory data restricted to MCS index 10, classical IDD improves upon the non-iterative LMMSE receiver, and pretrained DUIDD provides a further reduction in dataset BLER. Site-specific DUIDD finetuning reduces the dataset BLER by another 0.004 in absolute terms. The small additional gain is consistent with DUIDD having only 30 tunable parameters [[2]](#ref-2).

<p align="left">
  <img src="../../fig/duidd/duidd_ft-1.png" alt="Site-specific spatial, temporal, and frequency covariance-matrix heatmaps" width="720">
</p>


<a id="references"></a>

## References

<a id="ref-1"></a>[1] R. Wiesmayr, C. Dick, J. Hoydis, and C. Studer, “DUIDD: Deep-Unfolded Interleaved Detection and Decoding for MIMO Wireless Systems,” in *Proc. Asilomar Conference on Signals, Systems, and Computers*, 2022. Available: https://arxiv.org/abs/2212.07816

<a id="ref-2"></a>[2] R. Wiesmayr, N. B. Baytekin, C. Dick, and C. Studer, “On the Impact of Site-Specific Training for a Real-World 5G NR System,” in *Proc. Asilomar Conference on Signals, Systems, and Computers*, 2026.

<a id="ref-3"></a>[3] NVIDIA Corporation, “Aerial CUDA-Accelerated RAN,” release 25-2. Available: https://docs.nvidia.com/aerial/cuda-accelerated-ran/25-2/index.html
